In [ ]:
import os
import glob
import random
import re

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
from torchvision.utils import save_image
from torchvision import models

import sys
sys.path.append('/kaggle/working')

In [ ]:
class PartialConv2d(nn.Conv2d):
    def __init__(self, *args, **kwargs):
        if 'multi_channel' in kwargs:
            self.multi_channel = kwargs.pop('multi_channel')
        else:
            self.multi_channel = False

        if 'return_mask' in kwargs:
            self.return_mask = kwargs.pop('return_mask')
        else:
            self.return_mask = False

        super(PartialConv2d, self).__init__(*args, **kwargs)

        if self.multi_channel:
            self.weight_maskUpdater = torch.ones(self.out_channels, self.in_channels, self.kernel_size[0], self.kernel_size[1])
        else:
            self.weight_maskUpdater = torch.ones(1, 1, self.kernel_size[0], self.kernel_size[1])

        self.slide_winsize = self.weight_maskUpdater.shape[1] * self.weight_maskUpdater.shape[2] * self.weight_maskUpdater.shape[3]
        self.last_size = (None, None, None, None)
        self.update_mask = None
        self.mask_ratio = None

    def forward(self, input, mask_in=None):
        assert len(input.shape) == 4
        if mask_in is not None or self.last_size != tuple(input.shape):
            self.last_size = tuple(input.shape)
            with torch.no_grad():
                if self.weight_maskUpdater.type() != input.type():
                    self.weight_maskUpdater = self.weight_maskUpdater.to(input)
                mask = mask_in if mask_in is not None else torch.ones(1, 1, input.data.shape[2], input.data.shape[3]).to(input)
                self.update_mask = F.conv2d(mask, self.weight_maskUpdater, bias=None, stride=self.stride, padding=self.padding, dilation=self.dilation, groups=1)
                self.mask_ratio = self.slide_winsize / (self.update_mask + 1e-8)
                self.update_mask = torch.clamp(self.update_mask, 0, 1)
                self.mask_ratio = torch.mul(self.mask_ratio, self.update_mask)

        raw_out = super(PartialConv2d, self).forward(torch.mul(input, mask_in) if mask_in is not None else input)

        if self.bias is not None:
            bias_view = self.bias.view(1, self.out_channels, 1, 1)
            output = torch.mul(raw_out - bias_view, self.mask_ratio) + bias_view
            output = torch.mul(output, self.update_mask)
        else:
            output = torch.mul(raw_out, self.mask_ratio)

        if self.return_mask:
            return output, self.update_mask
        return output

In [ ]:
def gram_matrix(x):
    (b, ch, h, w) = x.size()
    features = x.view(b, ch, w * h)
    gram = torch.baddbmm(torch.zeros(b, ch, ch).to(x.device), features, features.transpose(1, 2), beta=0, alpha=1./(ch * h * w))
    return gram

class InpaintingLoss(nn.Module):
    def __init__(self, vgg_device):
        super().__init__()
        vgg16 = models.vgg16(pretrained=True).features.to(vgg_device).eval()
        self.enc_vgg = nn.ModuleList([vgg16[:4], vgg16[4:9], vgg16[9:16]])
        for p in self.parameters(): p.requires_grad = False
        self.l1 = nn.L1Loss()

    def forward(self, mask, output, gt):
        comp = mask * gt + (1 - mask) * output
        # L1 Losses
        l_valid = self.l1(mask * output, mask * gt)
        l_hole = self.l1((1 - mask) * output, (1 - mask) * gt)

        # VGG Losses
        out_feat = self.get_vgg_feat(output)
        gt_feat = self.get_vgg_feat(gt)
        comp_feat = self.get_vgg_feat(comp)

        l_perc = sum(self.l1(o, g) + self.l1(c, g) for o, g, c in zip(out_feat, gt_feat, comp_feat))
        l_style = sum(self.l1(gram_matrix(o), gram_matrix(g)) + self.l1(gram_matrix(c), gram_matrix(g)) for o, g, c in zip(out_feat, gt_feat, comp_feat))

        # TV Loss
        l_tv = torch.mean(torch.abs(comp[:, :, :, :-1] - comp[:, :, :, 1:])) + torch.mean(torch.abs(comp[:, :, :-1, :] - comp[:, :, 1:, :]))

        return l_valid + 6*l_hole + 0.05*l_perc + 120*l_style + 0.1*l_tv

    def get_vgg_feat(self, x):
        feats = []
        for layer in self.enc_vgg:
            x = layer(x)
            feats.append(x)
        return feats

In [ ]:
class PConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k, s, p, use_bn=True, act='relu'):
        super().__init__()
        self.pconv = PartialConv2d(in_ch, out_ch, k, s, p, multi_channel=True, return_mask=True)
        self.bn = nn.BatchNorm2d(out_ch) if use_bn else None
        self.act = nn.ReLU(inplace=True) if act=='relu' else nn.LeakyReLU(0.2, inplace=True) if act=='leaky' else None

    def forward(self, x, m):
        x, m = self.pconv(x, m)
        if self.bn: x = self.bn(x)
        if self.act: x = self.act(x)
        return x, m

class PConvUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = PConvBlock(3, 64, 7, 2, 3, use_bn=False)
        self.enc2 = PConvBlock(64, 128, 5, 2, 2)
        self.enc3 = PConvBlock(128, 256, 5, 2, 2)
        self.enc4 = PConvBlock(256, 512, 3, 2, 1)
        for i in range(5, 9): setattr(self, f'enc{i}', PConvBlock(512, 512, 3, 2, 1))

        # Decoder
        for i in range(9, 13): setattr(self, f'dec{i}', PConvBlock(512+512, 512, 3, 1, 1, act='leaky'))
        self.dec13 = PConvBlock(512+256, 256, 3, 1, 1, act='leaky')
        self.dec14 = PConvBlock(256+128, 128, 3, 1, 1, act='leaky')
        self.dec15 = PConvBlock(128+64, 64, 3, 1, 1, act='leaky')
        self.dec16 = PConvBlock(64+3, 3, 3, 1, 1, use_bn=False, act='none')

    def forward(self, x, m):
        e1, m1 = self.enc1(x, m)
        e2, m2 = self.enc2(e1, m1)
        e3, m3 = self.enc3(e2, m2)
        e4, m4 = self.enc4(e3, m3)
        e5, m5 = self.enc5(e4, m4)
        e6, m6 = self.enc6(e5, m5)
        e7, m7 = self.enc7(e6, m6)
        e8, m8 = self.enc8(e7, m7)

        def up(x, m, e, em):
            x, m = F.interpolate(x, scale_factor=2), F.interpolate(m, scale_factor=2)
            return torch.cat([x, e], 1), torch.cat([m, em], 1)

        d, dm = self.dec9(*up(e8, m8, e7, m7))
        d, dm = self.dec10(*up(d, dm, e6, m6))
        d, dm = self.dec11(*up(d, dm, e5, m5))
        d, dm = self.dec12(*up(d, dm, e4, m4))
        d, dm = self.dec13(*up(d, dm, e3, m3))
        d, dm = self.dec14(*up(d, dm, e2, m2))
        d, dm = self.dec15(*up(d, dm, e1, m1))
        out, _ = self.dec16(*up(d, dm, x, m))
        return out

In [ ]:
# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==========================================
IMG_DIR = '.../data/train_dataset'
MASK_TRAIN_DIR = '../data/train_mask/disocclusion_img_mask'
SAVE_DIR = '../checkpoint'

os.makedirs(SAVE_DIR, exist_ok=True)

BATCH_SIZE = 16
IMG_SIZE = 256
EPOCHS = 10

# ==========================================
# 2. TẢI DỮ LIỆU
# ==========================================
print("Đang chuẩn bị dữ liệu...")
train_images = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
    train_images.extend(glob.glob(os.path.join(IMG_DIR, ext)))


# ==========================================
# 3. DATASET VỚI AUTO-FIX MASK
# ==========================================
class TrainDataset(Dataset):
    def __init__(self, image_paths, mask_dir, img_size=256):
        self.image_paths = image_paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, '*.*'))

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        gt_img = Image.open(self.image_paths[idx]).convert('RGB')
        gt_img = self.img_transform(gt_img)

        mask_path = random.choice(self.mask_paths)
        mask = Image.open(mask_path).convert('L')
        mask = self.mask_transform(mask)

        # AUTO-FIX: Tự động chuẩn hóa Mask (Nền = 1, Rách = 0)
        mean_val = mask.mean().item()
        if mean_val > 0.5:
            # Nếu trắng chiếm đa số -> Nền trắng, Rách đen
            mask = (mask > 0.5).float()
        else:
            # Nếu đen chiếm đa số -> Nền đen, Rách trắng -> Cần lật lại
            mask = (mask < 0.5).float()

        mask = mask.repeat(3, 1, 1)
        
        return gt_img, mask

train_dataset = TrainDataset(train_images, MASK_TRAIN_DIR, img_size=IMG_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

# ==========================================
# 4. KHỞI TẠO MÔ HÌNH & TỰ ĐỘNG RESUME
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

model = PConvUNet().to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

criterion = InpaintingLoss(vgg_device=device).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

start_epoch = 0

def get_latest_checkpoint():
    # Quét thư mục Input (dữ liệu cũ) và Working (dữ liệu mới)
    # Lệnh ** cho phép quét sâu vào các thư mục con của Kaggle
    paths = glob.glob('/kaggle/input/**/pconv_celeba_epoch_*.pth', recursive=True) + \
            glob.glob(os.path.join(SAVE_DIR, 'pconv_celeba_epoch_*.pth'))
    
    if not paths:
        return None
    
    # Lọc lấy file có số Epoch lớn nhất
    paths.sort(key=lambda x: int(re.findall(r'\d+', os.path.basename(x))[-1]))
    return paths[-1]

latest_ckpt = get_latest_checkpoint()


if latest_ckpt:
    print(f"\nĐang nạp Checkpoint mới nhất từ: {latest_ckpt}")
    checkpoint = torch.load(latest_ckpt, map_location=device)

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
    else:
        state_dict = checkpoint
    
    # Nạp Model (Xử lý an toàn cho DataParallel)
    if isinstance(model, nn.DataParallel) and not list(state_dict.keys())[0].startswith('module.'):
        state_dict = {f'module.{k}': v for k, v in state_dict.items()}
    elif not isinstance(model, nn.DataParallel) and list(state_dict.keys())[0].startswith('module.'):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        
    model.load_state_dict(state_dict)

    print(f"Đã nạp thành công! Chuẩn bị train tiếp từ Epoch {start_epoch + 1}\n")
else:
    print("\Huấn luyện mới (Bắt đầu epoch 1)!\n")

# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN
# ==========================================
print("=== BẮT ĐẦU HUẤN LUYỆN ===")
for epoch in range(start_epoch, EPOCHS):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, (gt_imgs, masks) in enumerate(progress_bar):
        gt_imgs = gt_imgs.to(device)
        masks = masks.to(device)

        input_imgs = gt_imgs * masks

        optimizer.zero_grad()
        outputs = model(input_imgs, masks)
        loss = criterion(masks, outputs, gt_imgs)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

        # Lưu ảnh sample mỗi 5000 bước để xem trước
        if (i + 1) % 5000 == 0:
            comp_imgs = masks * gt_imgs + (1 - masks) * outputs
            comparison = torch.cat([input_imgs[:4], comp_imgs[:4], gt_imgs[:4]], dim=0)
            save_image(comparison, os.path.join(SAVE_DIR, f'sample_epoch{epoch+1}_step{i+1}.png'), nrow=4)


    # Lưu checkpoint chính thức sau mỗi epoch
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }
    checkpoint_path = os.path.join(SAVE_DIR, f'pconv_celeba_epoch_{epoch+1}.pth')
    torch.save(checkpoint, checkpoint_path)
    print(f"\nĐã lưu mô hình tại: {checkpoint_path}\n")